# Credit Card Fraud Detection (OpenShift AI + MLflow)

Same flow as the [AI on OpenShift demo](https://ai-on-openshift.io/demos/credit-card-fraud-detection-mlflow/credit-card-fraud/#3-train-the-model), with **ONNX export updated** for current workbench images (TensorFlow 2.16+ / Keras 3): SavedModel export + `tf2onnx` CLI instead of `tf2onnx.convert.from_keras` (which breaks with `keras_tensor_…` / `KeyError`).

**Data:** place `card_transdata.csv` in `../data/` (clone [credit-fraud-detection-demo](https://github.com/red-hat-data-services/credit-fraud-detection-demo) or copy the CSV from there).

In [ ]:
!pip install pip -qU
!pip install -r requirements.txt -q

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import class_weight
from tensorflow.keras.layers import Activation, BatchNormalization, Dense, Dropout
from tensorflow.keras.models import Sequential

import matplotlib.pyplot as plt
import mlflow
import onnx
import seaborn as sns

In [ ]:
mlflow.__version__

In [ ]:
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_TRACKING_AUTH"] = "kubernetes-namespaced"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [ ]:
def keras_model_to_onnx(model: tf.keras.Model, opset: int = 13) -> onnx.ModelProto:
    """Convert a trained Keras model to ONNX for OpenShift AI Model Serving.

    Uses a SavedModel directory + the supported ``python -m tf2onnx.convert`` CLI
    (avoids ``tf2onnx.convert.from_keras`` issues on TF 2.16+ / Keras 3).
    Note: tf2onnx expects ``--signature_def`` (underscore), not ``--signature-def``.
    """
    export_dir = tempfile.mkdtemp(prefix="cc_fraud_savedmodel_")
    onnx_fd, onnx_path = tempfile.mkstemp(suffix=".onnx")
    os.close(onnx_fd)
    try:
        if hasattr(model, "export"):
            model.export(export_dir)
        else:
            tf.saved_model.save(model, export_dir)

        loaded = tf.saved_model.load(export_dir)
        sig_keys = list(loaded.signatures.keys())
        if not sig_keys:
            raise RuntimeError("SavedModel has no signatures; cannot convert to ONNX.")
        if "serve" in sig_keys:
            signature = "serve"
        elif "serving_default" in sig_keys:
            signature = "serving_default"
        else:
            signature = sig_keys[0]

        cmd = [
            sys.executable,
            "-m",
            "tf2onnx.convert",
            "--saved-model",
            export_dir,
            "--output",
            onnx_path,
            "--opset",
            str(opset),
            "--signature_def",
            signature,
        ]
        proc = subprocess.run(cmd, capture_output=True, text=True)
        if proc.returncode != 0:
            raise RuntimeError(
                f"tf2onnx.convert failed (exit {proc.returncode}). Command:\n"
                f"{' '.join(cmd)}\n\nstderr:\n{proc.stderr}\nstdout:\n{proc.stdout}"
            )
        return onnx.load(onnx_path)
    finally:
        shutil.rmtree(export_dir, ignore_errors=True)
        try:
            os.remove(onnx_path)
        except OSError:
            pass

### Load data

In [ ]:
# Load the CSV data which we will use to train the model.
# It contains the following fields:
#   distancefromhome - The distance from home where the transaction happened.
#   distancefromlast_transaction - The distance from last transaction happened.
#   ratiotomedianpurchaseprice - Ratio of purchased price compared to median purchase price.
#   repeat_retailer - If it's from a retailer that already has been purchased from before.
#   used_chip - If the (credit card) chip was used.
#   usedpinnumber - If the PIN number was used.
#   online_order - If it was an online order.
#   fraud - If the transaction is fraudulent.

Data = pd.read_csv('../data/card_transdata.csv')
Data.head()

### Train / validation / test split and scaling

In [ ]:
# Set the input (X) and output (Y) data. 
# The only output data we have is if it's fraudulent or not, and all other fields go as inputs to the model.

X = Data.drop(columns = ['fraud'])
y = Data['fraud']

# Split the data into training and testing sets so we have something to test the trained model with.

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, stratify = y)

X_train, X_val, y_train, y_val = train_test_split(X_train,y_train, test_size = 0.2, stratify = y_train)

# Scale the data to remove mean and have unit variance. This means that the data will be between -1 and 1, which makes it a lot easier for the model to learn than random potentially large values.
# It is important to only fit the scaler to the training data, otherwise you are leaking information about the global distribution of variables (which is influenced by the test set) into the training set.

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

# Since the dataset is unbalanced (it has many more non-fraud transactions than fraudulent ones), we set a class weight to weight the few fraudulent transactions higher than the many non-fraud transactions.

class_weights = class_weight.compute_class_weight('balanced',classes = np.unique(y_train),y = y_train)
class_weights = {i : class_weights[i] for i in range(len(class_weights))}

y_train = y_train.to_numpy()
y_val = y_val.to_numpy()
y_test = y_test.to_numpy()

### Build the DNN (same architecture as the original demo)

In [ ]:
# Build the model, the model we build here is a simple fully connected deep neural network, containing 3 hidden layers and one output layer.

model = Sequential()
model.add(Dense(32, name='dense', activation = 'relu', input_dim = len(X.columns)))
model.add(Dropout(0.2))
model.add(Dense(32, name='dense_02'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Dense(32, name='dense_03'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Dense(1, name='dense_04', activation = 'sigmoid'))
model.compile(optimizer='SGD',loss='binary_crossentropy',metrics=['accuracy'])
model.summary()

### Configure MLFlow
### Train the model, plot the confusion matrix and push the artifacts to MLFlow.

In [ ]:
# Autolog metrics/params only — do not log or register the TensorFlow/Keras artifact here.
# ONNX is the deployable format for OpenShift AI model serving; we register it manually below.
mlflow.set_experiment("credit-card-fraud")
mlflow.tensorflow.autolog(log_models=False)

### Train, log metrics, export ONNX to MLflow

**Changed from the published demo:** replace `tf2onnx.convert.from_keras(model)` with `keras_model_to_onnx(model)` so ONNX is produced reliably on current OpenShift AI images.

**MLflow layout:** the ONNX bundle is logged under the run artifact **`credit-card-fraud-onnx`** (not the generic name `models`) and registered as **`credit-card-fraud-onnx`**. When you deploy from object storage in OpenShift AI, use the folder path that ends with `artifacts/credit-card-fraud-onnx/`.

In [ ]:
with mlflow.start_run():
    epochs = 2
    model.fit(
        X_train,
        y_train,
        epochs=epochs,
        validation_data=(scaler.transform(X_val), y_val),
        verbose=True,
        class_weight=class_weights,
    )

    y_pred_temp = model.predict(scaler.transform(X_test), verbose=0)
    threshold = 0.995
    y_pred = np.where(y_pred_temp > threshold, 1, 0)
    c_matrix = confusion_matrix(y_test, y_pred)

    ax = sns.heatmap(c_matrix, annot=True, cbar=False, cmap="Blues")
    ax.set_xlabel("Prediction")
    ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix")
    plt.show()

    t_n, f_p, f_n, t_p = c_matrix.ravel()
    mlflow.log_metric("tn", int(t_n))
    mlflow.log_metric("fp", int(f_p))
    mlflow.log_metric("fn", int(f_n))
    mlflow.log_metric("tp", int(t_p))

    model_proto = keras_model_to_onnx(model)
    # Same label for artifact folder (under the run) and Model Registry entry.
    name = "credit-card-fraud-onnx"
    mlflow.onnx.log_model(
        model_proto,
        artifact_path=name,
        registered_model_name=name,
    )